# Расчёт размера выборки перед экспериментом

Продакт пришёл с задачей:

> «Запускаем тест новой подачи цены. Хочу быть уверен, что мы поймаем даже эффект в **0.5 п.п.** по конверсии в заказ»

**п.п. = процентный пункт**: рост конверсии с 30.0% до 30.5% — это +0.5 п.п. (но +1.7% относительных). Не путай — от этого зависит весь расчёт.

### Какую метрику считаем

В этом ноутбуке мы разбираем расчёт для **метрики-доли** — конверсии `screen_view → order_confirm`. Это самый частый и самый простой случай. Но метрики бывают разными, и расчёт выборки для них отличается:

- **доля / конверсия** (наш случай) — есть аналитическая формула;
- **среднее на пользователя** (поездки, выручка) — формула тоже есть, но нужна дисперсия метрики, а у денежных метрик — тяжёлые хвосты;
- **ratio-метрики** (отмены на заказ) — юнит анализа мельче юнита рандомизации, простые формулы врут;
- **абсолютные** (общая выручка, число заказов) — считаются через связку с пользовательскими.

Логика при этом всегда одна: базовое значение метрики → MDE → alpha и мощность → размер выборки → сверка с реальной аудиторией.

### План ноутбука

1. Достать базовую конверсию из исторических данных.
2. Посчитать нужный размер выборки **аналитической формулой**.
3. Перепроверить **симуляциями** (Monte Carlo).
4. Сходить в базу и выяснить, **есть ли у нас столько аудитории** — и дать продакту честный ответ.

## Как Python ходит в базы данных

В работе аналитика Python и SQL живут в связке: тяжёлые агрегации по миллионам строк делает **база данных**, а Python забирает уже маленький результат и делает с ним статистику, графики, тесты. Тащить сырые события в pandas — плохая привычка: то, что база агрегирует за миллисекунды, Python будет жевать минутами.

Подключение из Python устроено одинаково почти для любой базы — это стандарт **DB-API**, и конструкция всегда такая:

```
соединение = connect(адрес, логин, пароль)   # 1. подключились
cursor = соединение.cursor()                  # 2. взяли курсор
cursor.execute("SELECT ...")                 # 3. выполнили запрос (обычная строка!)
строки = cursor.fetchall()                    # 4. забрали результат
df = pd.DataFrame(строки, columns=...)        # 5. положили в датафрейм
```

Меняется только библиотека: `psycopg2` для PostgreSQL, `clickhouse-connect` для ClickHouse, `sqlite3` встроен в Python, для MySQL/Trino/BigQuery — свои. Выучил конструкцию один раз — умеешь работать с любой базой.

In [ ]:
# pip install clickhouse-connect pandas numpy scipy statsmodels
import numpy as np
import pandas as pd
from clickhouse_connect.dbapi import connect

conn = connect(host="localhost", port=8123, username="default", password="platform")
cursor = conn.cursor()

# Шаги 3-4-5 конструкции удобно завернуть в функцию — дальше будем пользоваться ей
def q(sql: str) -> pd.DataFrame:
    cursor.execute(sql)
    columns = [col[0] for col in cursor.description]
    return pd.DataFrame(cursor.fetchall(), columns=columns)

# Проверка подключения: сколько каких событий в базе
q("""
SELECT event_name, count() AS events
FROM ab.events
GROUP BY event_name
ORDER BY events DESC
""")

## Шаг 1. Базовая конверсия из истории

Для формулы нужна `p_base` — базовая конверсия `screen_view → order_confirm` на пользователя в день, по **аудитории будущего эксперимента** (тот же регион и фильтры, что будут в карточке).

Подумай перед запросом: почему считаем по аудитории эксперимента, а не по всем пользователям? Что будет с расчётом, если базовую конверсию взять по другому региону?

In [ ]:
# TODO: SQL по ab.events за предпериод:
#   доля пользователей с order_confirm среди пользователей со screen_view
#   (на пользователя в день, по своему региону)

p_base = ...   # TODO: одно число, например 0.31

## Шаг 2. Аналитический калькулятор

Для сравнения двух долей размер выборки **на группу**:

$$n = \frac{2 \, (z_{1-\alpha/2} + z_{1-\beta})^2 \; \bar{p}(1-\bar{p})}{\delta^2}$$

где $\delta$ — MDE в долях (0.5 п.п. = 0.005), $\bar{p}$ — средняя конверсия двух групп, $z$ — квантили нормального распределения.

Напоминание терминов:
- **alpha** — допустимая вероятность ложного срабатывания (обычно 0.05);
- **power = 1 − beta** — вероятность поймать эффект, если он есть (обычно 0.8);
- **MDE** — минимальный эффект, который хотим уметь детектировать.

In [ ]:
from scipy import stats

def required_sample_size(p_base: float, mde_pp: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Размер выборки НА ГРУППУ для z-теста двух долей."""
    # TODO: квантили возьми из stats.norm.ppf
    ...

# TODO: посчитай n для сетки MDE: 0.5, 1, 2, 3 п.п. и сведи в таблицу:
# MDE | n на группу | всего

## Шаг 3. Проверка симуляциями (Monte Carlo)

Формула выше выведена для идеальной доли: независимые наблюдения, нормальное приближение. Реальные данные устроены сложнее, поэтому **в индустрии симуляции — стандарт**: если есть вычислительные ресурсы, MDE, alpha и мощность проверяют симуляциями, потому что они показывают **поведение самих данных, а не наших предположений о них**. Формула молчит про тяжёлые хвосты, зависимость наблюдений и кривое логирование — симуляция на исторических данных всё это честно воспроизводит. А для метрик, где формулы просто нет (ratio, средние с дикой дисперсией), симуляция — вообще единственный надёжный путь.

Алгоритм оценки мощности:
1. Возьми пользователей из исторических данных с их фактическим исходом (была конверсия / нет).
2. Много раз (например, 2000): сэмплируй две группы по `n` пользователей, группе B искусственно «подними» конверсию на MDE, прогони стат-тест, запомни p-value.
3. Мощность = доля итераций, где p-value < alpha. На размере из формулы она должна сойтись с заявленной (0.8).

Бонус: проверь и alpha — прогони то же самое **без** инъекции эффекта. Доля ложных срабатываний должна быть ≈ 0.05. Это, по сути, A/A-тест на исторических данных.

In [ ]:
def simulate_power(user_outcomes: np.ndarray, n_per_group: int, mde_pp: float,
                   alpha: float = 0.05, n_sims: int = 2000, seed: int = 42) -> float:
    """Доля значимых результатов при истинном эффекте = mde_pp."""
    rng = np.random.default_rng(seed)
    significant = 0
    for _ in range(n_sims):
        # TODO: сэмплируй группы A и B из user_outcomes
        # TODO: инъекция эффекта в B: часть нулей переверни в единицы с нужной вероятностью
        # TODO: z-тест двух долей (statsmodels.stats.proportion.proportions_ztest)
        ...
    return significant / n_sims

# TODO: сравни мощность из симуляций с формулой на 2-3 значениях MDE

## Шаг 4. А есть ли у нас столько аудитории?

Формула сказала, сколько пользователей **нужно**. Теперь вопрос, сколько их **есть**. Эксперимент ты ещё не заводил, сплитовалку не трогал — и это правильно: аналитик оценивает реализуемость **до** того, как что-то настраивать. Всё, что нужно, уже лежит в базе.

Посчитай SQL-запросами по своим фильтрам аудитории (регион, активность):

1. Сколько **уникальных** пользователей появляется в данных за последний день? За 7 дней? За 14?
2. Построй кумулятивную кривую: сколько НОВЫХ уников добавляет каждый следующий день. Почему уников за 14 дней не в 14 раз больше, чем за день? Что это значит для длительности набора?
3. Сравни с расчётом: нужно `n × число групп` пользователей. За сколько дней столько наберётся? А для MDE 0.5 п.п.?

In [ ]:
# TODO: uniqExact(user_id) по окнам 1 / 7 / 14 последних дней по своему региону

# TODO: кумулятивная кривая набора уников по дням
#   (подсказка: минимальная дата появления каждого пользователя + группировка по ней)

# TODO: вывод: дней набора для каждого MDE из таблицы шага 2

### Вопросы на самопроверку

1. Хватит ли аудитории на MDE 0.5 п.п. за разумный срок? Сколько недель понадобилось бы?
2. Какие у продакта есть варианты, если не хватает? Назови минимум три и издержки каждого.
3. Почему нельзя просто «подержать тест подольше, пока не станет значимо»?
4. Что изменится в расчёте, если целевой метрикой сделать `trips_per_user` (среднее), а не конверсию (долю)?

### Формат сдачи

Короткое сообщение продакту (3–6 предложений): реализуемо ли MDE 0.5 п.п., что предлагаешь вместо этого и почему. Плюс таблица MDE → размер выборки → длительность набора. Числа перенеси в раздел 8 карточки эксперимента и отправь ментору обновлённую версию.